# BAN 612 — Week 2 In-Class Practice
## Clinic Scheduling Data: Formats, Data Types, Loading, and Validation

You are working with a healthcare scheduling system. The goal is to practice the Week 2 skills on a new context:

- loading CSV, Excel, and JSON files;
- inspecting schemas and data types;
- standardizing text and dates;
- identifying missing and invalid values;
- checking unique keys and referential integrity;
- validating business rules across multiple columns;
- documenting what should be corrected versus merely flagged.

> **Business setting:** A multi-clinic medical group has combined scheduling data from several systems. Before management analyzes no-shows, wait times, or telehealth utilization, the data must be validated.

## 1. Import the libraries

### What this code does
`pandas` is the main library we will use for tabular data. We import it using the conventional alias `pd`. `json` is part of Python's standard library and is useful when we want to inspect raw JSON structures.

### Syntax
```python
import pandas as pd
import json
```

`import package as alias` loads a package and gives it a shorter name. After `import pandas as pd`, we call functions such as `pd.read_csv()` rather than typing `pandas.read_csv()` each time.

In [1]:
import pandas as pd
import json

## 2. Define file paths

### What this code does
We store file names in variables. This makes later code easier to read and change.

A Python string is written inside quotes. Raw strings such as `r"..."` are especially useful for Windows paths because backslashes are treated literally.

If the data files are in the **same folder as this notebook**, simple relative paths are sufficient.

In [2]:
appointments_file = 'clinic_appointments.csv'
patients_file = 'clinic_patients.xlsx'
providers_file = 'clinic_providers.json'

## 3. Load a CSV file with `pd.read_csv()`

### General syntax
```python
pd.read_csv(filepath_or_buffer, sep=',', header='infer', dtype=None, na_values=None, usecols=None)
```

### Important parameters
- `filepath_or_buffer`: file path or URL.
- `sep`: delimiter. Default is a comma. Use `sep='\t'` for tab-separated data.
- `header`: row containing column names. The default `'infer'` normally uses the first row.
- `dtype`: force one or more columns to a specific type. Example: `dtype={'patient_id': 'string'}`.
- `na_values`: additional values to interpret as missing, such as `['NA', 'unknown', '999']`.
- `usecols`: load only selected columns.

For the first read, we intentionally allow Pandas to infer the types so that we can inspect what it decided.

In [3]:
appointments = pd.read_csv(appointments_file)
appointments.head()

,appointment_id,patient_id,provider_id,booked_at,appointment_at,status,visit_type,checkin_at,checkout_at,telehealth
0,A3001,P1011,D201,2026-07-17 11:15,2026-08-01 12:15,Completed,New Patient,2026-08-01 12:26,2026-08-01 13:28,No
1,A3002,P1001,D201,2026-07-01 07:15,2026-08-03 11:15,Completed,Urgent,2026-08-03 11:22,2026-08-03 11:49,Yes
2,A3003,P1008,D205,2026-07-12 06:15,2026-08-09 08:15,no-show,Annual Physical,2026-08-09 08:09,2026-08-09 08:37,Yes
3,A3004,P1002,D204,2026-07-18 13:30,2026-08-04 13:30,No Show,New Patient,NaN,NaN,No
4,A3005,P1002,D205,2026-08-05 13:15,2026-08-10 13:15,Completed,New Patient,2026-08-10 13:29,2026-08-10 14:02,Yes


In [4]:
appointments.sample(5,random_state=42)

,appointment_id,patient_id,provider_id,booked_at,appointment_at,status,visit_type,checkin_at,checkout_at,telehealth
19,A3020,P1013,D204,2026-08-07 06:00,2026-08-14 11:00,No Show,Annual Physical,NaN,NaN,No
16,A3017,P1012,D203,2026-07-10 10:45,2026-08-13 13:45,Completed,New Patient,2026-08-13 13:42,2026-08-13 14:11,No
15,A3015,P1011,D204,2026-07-15 06:30,2026-08-18 10:30,No Show,Follow-up,NaN,NaN,Yes
26,A3027,P1011,D202,2026-08-10 05:15,2026-08-13 09:15,Completed,Annual Physical,2026-08-13 09:18,2026-08-13 10:15,No
4,A3005,P1002,D205,2026-08-05 13:15,2026-08-10 13:15,Completed,New Patient,2026-08-10 13:29,2026-08-10 14:02,Yes


## 4. Load an Excel file with `pd.read_excel()`

### General syntax
```python
pd.read_excel(io, sheet_name=0, header=0, usecols=None, dtype=None)
```

### Important parameters
- `io`: Excel file path.
- `sheet_name=0`: read the first worksheet. You can also use a sheet name such as `sheet_name='Patients'`.
- `sheet_name=None`: return a dictionary containing **all** worksheets.
- `usecols='A:F'`: restrict the columns loaded.
- `dtype`: optionally force data types.

Excel files may contain multiple sheets, formatting, formulas, and mixed types, so inspection after loading is essential.

In [5]:
patients = pd.read_excel(patients_file)
patients.head()

,patient_id,patient_name,date_of_birth,sex,insurance,zip_code
0,P1001,Ava Patel,1987-04-12,F,Commercial,94588
1,P1002,Noah Kim,1975-11-03,M,Medicare,94566
2,P1003,Mia Garcia,1999-08-21,F,Medi-Cal,94538
3,P1004,Liam Chen,1962-01-19,M,Medicare,94550
4,P1005,Emma Johnson,1982-06-30,F,Commercial,9458


## 5. Load a JSON file

`pd.read_json()` can read many JSON files directly into a DataFrame.

### General syntax
```python
pd.read_json(path_or_buf, orient=None, lines=False)
```

### Useful parameters
- `orient`: describes how records are organized. Common values include `'records'`, `'columns'`, and `'index'`.
- `lines=True`: used for **JSON Lines / NDJSON**, where each line is a separate JSON object.

Our provider file is a simple list of records, so Pandas can read it directly.

In [6]:
providers = pd.read_json(providers_file)
providers

,provider_id,provider_name,specialty,location
0,D201,Dr. Rivera,Primary Care,Pleasanton
1,D202,Dr. Singh,Cardiology,Hayward
2,D203,Dr. Lee,Endocrinology,Fremont
3,D204,Dr. Nguyen,Primary Care,Pleasanton
4,D205,Dr. Carter,Pediatrics,Hayward


### Optional: inspect the raw JSON structure

Sometimes API responses or JSON files are nested and cannot immediately be converted into a clean table. In those cases, inspect the raw Python object first.

`json.load(file_object)` reads JSON from an open file and returns Python dictionaries/lists.

In [7]:
with open(providers_file, 'r') as f:
    provider_raw = json.load(f)

type(provider_raw), provider_raw[:2]

(list,
 [{'provider_id': 'D201',
   'provider_name': 'Dr. Rivera',
   'specialty': 'Primary Care',
   'location': 'Pleasanton'},
  {'provider_id': 'D202',
   'provider_name': 'Dr. Singh',
   'specialty': 'Cardiology',
   'location': 'Hayward'}])

## 6. Inspect dimensions, columns, and data types

Three useful Pandas attributes/methods are:

- `df.shape` → tuple `(number_of_rows, number_of_columns)`
- `df.columns` → column labels
- `df.info()` → concise summary containing non-null counts and inferred data types

`info()` is a **method**, so it requires parentheses. `shape` is an **attribute**, so it does not.

In [8]:
print('Appointments shape:', appointments.shape)
print('Patients shape:', patients.shape)
print('Providers shape:', providers.shape)

appointments.info()

Appointments shape: (40, 10)
Patients shape: (15, 6)
Providers shape: (5, 4)
<class 'pandas.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   appointment_id  40 non-null     str  
 1   patient_id      40 non-null     str  
 2   provider_id     40 non-null     str  
 3   booked_at       40 non-null     str  
 4   appointment_at  40 non-null     str  
 5   status          40 non-null     str  
 6   visit_type      40 non-null     str  
 7   checkin_at      29 non-null     str  
 8   checkout_at     28 non-null     str  
 9   telehealth      40 non-null     str  
dtypes: str(10)
memory usage: 6.8 KB


## 7. Why did Pandas call our dates `object`?

CSV files do not carry rich data-type metadata. A date often arrives as text. Pandas therefore may use `object` or `string` until we explicitly parse the column.

Use `pd.to_datetime()` to convert text into Pandas datetime values.

### General syntax
```python
pd.to_datetime(arg, errors='raise', format=None, dayfirst=False)
```

### Important parameters
- `arg`: Series, list, or scalar to convert.
- `errors='raise'`: invalid dates produce an error.
- `errors='coerce'`: invalid dates become `NaT` (Not-a-Time). Very useful for auditing dirty data.
- `format='%Y-%m-%d'`: specify a known format for stricter parsing.
- `dayfirst=True`: interpret ambiguous dates such as `04/05/2026` as day/month/year. Use carefully.

For data-quality work, `errors='coerce'` lets us identify values that failed conversion.

In [9]:
appointments['appointment_at_parsed'] = pd.to_datetime(
    appointments['appointment_at'],
    errors='coerce',
    format='mixed'
)

appointments[['appointment_at', 'appointment_at_parsed']].head(12)

,appointment_at,appointment_at_parsed
0,2026-08-01 12:15,2026-08-01 12:15:00
1,2026-08-03 11:15,2026-08-03 11:15:00
2,2026-08-09 08:15,2026-08-09 08:15:00
3,2026-08-04 13:30,2026-08-04 13:30:00
4,2026-08-10 13:15,2026-08-10 13:15:00
5,2026-08-13 12:45,2026-08-13 12:45:00
6,2026-08-18 11:15,2026-08-18 11:15:00
7,08/09/2026 2:30 PM,2026-08-09 14:30:00
8,2026-08-13 15:15,2026-08-13 15:15:00
9,2026-08-13 13:15,2026-08-13 13:15:00


## 8. Find values that failed date parsing

`Series.isna()` returns `True` where a value is missing (`NaN`, `NaT`, or `None`).

Boolean indexing uses a True/False condition inside square brackets:
```python
df[condition]
```

The condition below means:
1. parsed date is missing, **and**
2. original text was not missing.

This distinguishes an intentionally blank field from a malformed date string.

In [10]:
bad_appointment_dates = appointments[
    appointments['appointment_at_parsed'].isna() &
    appointments['appointment_at'].notna()
]

bad_appointment_dates[['appointment_id','appointment_at']]

,appointment_id,appointment_at
37,A3038,2026-13-40 25:00


## 9. Parse several datetime columns efficiently

A `for` loop lets us repeat the same transformation for several columns.

The suffix `_dt` helps us keep the original text and the parsed value side-by-side during auditing. Keeping the raw field is often safer than overwriting it immediately.

In [11]:
datetime_cols = ['booked_at', 'appointment_at', 'checkin_at', 'checkout_at']

for col in datetime_cols:
    appointments[col + '_dt'] = pd.to_datetime(
        appointments[col],
        errors='coerce',
        format='mixed'
    )

appointments[[c + '_dt' for c in datetime_cols]].head()

,booked_at_dt,appointment_at_dt,checkin_at_dt,checkout_at_dt
0,2026-07-17 11:15:00,2026-08-01 12:15:00,2026-08-01 12:26:00,2026-08-01 13:28:00
1,2026-07-01 07:15:00,2026-08-03 11:15:00,2026-08-03 11:22:00,2026-08-03 11:49:00
2,2026-07-12 06:15:00,2026-08-09 08:15:00,2026-08-09 08:09:00,2026-08-09 08:37:00
3,2026-07-18 13:30:00,2026-08-04 13:30:00,NaT,NaT
4,2026-08-05 13:15:00,2026-08-10 13:15:00,2026-08-10 13:29:00,2026-08-10 14:02:00


## 10. Inspect categorical values with `value_counts()`

### General syntax
```python
Series.value_counts(normalize=False, dropna=True, sort=True)
```

### Important parameters
- `normalize=False`: return counts. Set `True` for proportions.
- `dropna=True`: exclude missing values. Set `False` to include them.
- `sort=True`: sort by frequency.

This is one of the fastest ways to discover inconsistent category labels.

In [12]:
appointments['status'].value_counts(dropna=False)

status
Completed     25
No Show        8
Cancelled      4
no-show        1
NS             1
Completed      1
Name: count, dtype: int64

## 11. Standardize text safely

Pandas string methods are accessed through the `.str` namespace.

Useful methods include:
- `.str.strip()` → remove leading/trailing spaces
- `.str.lower()` → lowercase
- `.str.upper()` → uppercase
- `.str.title()` → title case
- `.str.replace(old, new, regex=False)` → replace literal text

`regex=False` means the first argument is treated as **ordinary literal text**, not a regular-expression pattern.

In [13]:
appointments['status_clean'] = (
    appointments['status']
    .astype('string')
    .str.strip()
    .str.lower()
)

appointments[['status','status_clean']].drop_duplicates().sort_values('status_clean')

,status,status_clean
12,Cancelled,cancelled
0,Completed,completed
23,Completed,completed
3,No Show,no show
2,no-show,no-show
5,NS,ns


## 12. Map multiple labels to one standard category

`Series.replace(mapping_dictionary)` is convenient when a finite set of known values must be recoded.

Here, `no-show`, `no show`, and `ns` should all represent the same business status.

This is different from `.str.replace()`: `Series.replace()` replaces **whole values**, whereas `.str.replace()` works inside strings.

In [14]:
status_map = {
    'no-show': 'No Show',
    'no show': 'No Show',
    'ns': 'No Show',
    'completed': 'Completed',
    'cancelled': 'Cancelled'
}

appointments['status_standard'] = appointments['status_clean'].replace(status_map)
appointments[['status','status_standard']].drop_duplicates()

,status,status_standard
0,Completed,Completed
2,no-show,No Show
3,No Show,No Show
5,NS,No Show
12,Cancelled,Cancelled
23,Completed,Completed


## 13. Check for duplicate keys

A business key such as `appointment_id` should usually identify exactly one appointment.

### General syntax
```python
Series.duplicated(keep='first')
```

### `keep` options
- `'first'` → first occurrence is False; later duplicates are True.
- `'last'` → last occurrence is False.
- `False` → **all occurrences** of duplicated values are True.

For auditing, `keep=False` is useful because it shows every record involved in the duplicate.

In [15]:
duplicate_appts = appointments[
    appointments['appointment_id'].duplicated(keep=False)
].sort_values('appointment_id')

duplicate_appts

,appointment_id,patient_id,provider_id,booked_at,appointment_at,status,visit_type,checkin_at,checkout_at,telehealth,appointment_at_parsed,booked_at_dt,appointment_at_dt,checkin_at_dt,checkout_at_dt,status_clean,status_standard
14,A3015,P1004,D201,2026-07-07 09:00,2026-08-08 09:00,Cancelled,Follow-up,NaN,NaN,Yes,2026-08-08 09:00:00,2026-07-07 09:00:00,2026-08-08 09:00:00,NaT,NaT,cancelled,Cancelled
15,A3015,P1011,D204,2026-07-15 06:30,2026-08-18 10:30,No Show,Follow-up,NaN,NaN,Yes,2026-08-18 10:30:00,2026-07-15 06:30:00,2026-08-18 10:30:00,NaT,NaT,no show,No Show


## 14. Referential integrity: do patient IDs exist in the patient master?

`.isin(values)` checks whether each value appears in another collection. It returns a Boolean Series.

Before comparing IDs, we strip whitespace so that `' P1007 '` and `'P1007'` are treated consistently.

In [16]:
appointments['patient_id_clean'] = appointments['patient_id'].astype('string').str.strip()
patients['patient_id_clean'] = patients['patient_id'].astype('string').str.strip()

unknown_patients = appointments[
    ~appointments['patient_id_clean'].isin(patients['patient_id_clean'])
]

unknown_patients[['appointment_id','patient_id']]

,appointment_id,patient_id
10,A3011,P9999


## 15. Referential integrity for provider IDs

The tilde `~` means logical NOT for a Boolean Series.

So:
```python
~series.isin(valid_values)
```
means “values that are **not** in the valid list.”

In [17]:
unknown_providers = appointments[
    ~appointments['provider_id'].isin(providers['provider_id'])
]

unknown_providers[['appointment_id','provider_id']]

,appointment_id,provider_id
12,A3013,D299


## 16. Validate chronology: booking should not occur after the appointment

Once timestamps are true datetime objects, we can compare them directly.

`booked_at_dt > appointment_at_dt` produces `True` for logically impossible records.

In [18]:
booking_after_appointment = appointments[
    appointments['booked_at_dt'] > appointments['appointment_at_dt']
]

booking_after_appointment[[
    'appointment_id','booked_at','appointment_at'
]]

,appointment_id,booked_at,appointment_at
27,A3028,2026-08-20 12:00,2026-08-18 09:00


## 17. Validate chronology: checkout should occur after check-in

For completed visits, checkout time should normally be later than check-in time.

Notice that this validation rule involves **multiple columns**. Data quality is not limited to checking one variable at a time.

In [19]:
bad_visit_times = appointments[
    (appointments['status_standard'] == 'Completed') &
    appointments['checkin_at_dt'].notna() &
    appointments['checkout_at_dt'].notna() &
    (appointments['checkout_at_dt'] < appointments['checkin_at_dt'])
]

bad_visit_times[[
    'appointment_id','checkin_at','checkout_at'
]]

,appointment_id,checkin_at,checkout_at
24,A3025,2026-08-12 10:15,2026-08-12 09:50


## 18. Conditional completeness rule

A completed visit should normally have both check-in and checkout timestamps.

The `|` operator means OR between Boolean Series, while `&` means AND. Parentheses are important because they make the intended logical grouping explicit.

In [20]:
completed_missing_times = appointments[
    (appointments['status_standard'] == 'Completed') &
    (appointments['checkin_at_dt'].isna() | appointments['checkout_at_dt'].isna())
]

completed_missing_times[[
    'appointment_id','status','checkin_at','checkout_at'
]]

,appointment_id,status,checkin_at,checkout_at
30,A3031,Completed,2026-08-15 13:05,NaN


## 19. Another business rule: cancelled visits should not have a check-in time

Business rules depend on context. A value can be perfectly valid by itself but inconsistent with another field.

This is why data validation should encode **domain logic**, not merely check for missing values.

In [21]:
cancelled_with_checkin = appointments[
    (appointments['status_standard'] == 'Cancelled') &
    appointments['checkin_at_dt'].notna()
]

cancelled_with_checkin[['appointment_id','status','checkin_at']]

,appointment_id,status,checkin_at
33,A3034,Cancelled,2026-08-16 08:55


## 20. Standardize another category: telehealth

Real systems often encode Yes/No in several ways: `Yes`, `Y`, `1`, `TRUE`, etc.

We can first normalize text, then map the accepted variants to a common representation.

In [22]:
appointments['telehealth_clean'] = (
    appointments['telehealth']
    .astype('string')
    .str.strip()
    .str.lower()
    .replace({'y':'Yes', 'yes':'Yes', 'n':'No', 'no':'No'})
)

appointments[['telehealth','telehealth_clean']].drop_duplicates()

,telehealth,telehealth_clean
0,No,No
1,Yes,Yes
18,Y,Yes


## 21. Inspect patient-master quality

The appointment table is not the only source that can contain problems. Always audit the **reference/master data** too.

Here we inspect insurance and sex categories and check ZIP-code structure.

In [23]:
print(patients['insurance'].value_counts(dropna=False))
print()
print(patients['sex'].value_counts(dropna=False))

insurance
Commercial    8
Medicare      3
Medi-Cal      3
medicare      1
Name: count, dtype: int64

sex
F         7
M         7
Female    1
Name: count, dtype: int64


## 22. Use regex to validate a ZIP code pattern

Regular expressions (regex) describe text patterns.

The pattern `^\d{5}$` means:
- `^` → beginning of the string
- `\d` → digit
- `{5}` → exactly five repetitions
- `$` → end of the string

`Series.str.fullmatch(pattern)` checks whether the **entire string** matches the regex.

In [24]:
zip_is_valid = patients['zip_code'].astype('string').str.fullmatch(r'\d{5}')
patients.loc[~zip_is_valid, ['patient_id','zip_code']]

,patient_id,zip_code
4,P1005,9458


## 23. Parse patient dates of birth and compute approximate age

`pd.Timestamp.today()` returns the current timestamp. Subtracting datetime values gives timedeltas.

Dividing days by `365.25` gives an approximate age suitable for a data-quality check. For exact legal/clinical age, use a more precise date-based calculation.

In [25]:
patients['dob_dt'] = pd.to_datetime(
    patients['date_of_birth'],
    errors='coerce',
    format='mixed'
)

today = pd.Timestamp.today().normalize()
patients['approx_age'] = ((today - patients['dob_dt']).dt.days / 365.25).round(1)
patients[['patient_id','date_of_birth','approx_age']].head()

,patient_id,date_of_birth,approx_age
0,P1001,1987-04-12,39.4
1,P1002,1975-11-03,50.8
2,P1003,1999-08-21,27.0
3,P1004,1962-01-19,64.6
4,P1005,1982-06-30,44.2


## 24. Merge appointment and patient data with validation

### General syntax
```python
left.merge(right, how='inner', on='key', validate=None, indicator=False)
```

### Important parameters
- `how='inner'`: keep only matching keys. Other common choices: `'left'`, `'right'`, `'outer'`.
- `on='patient_id'`: common key column.
- `validate='many_to_one'`: assert that many appointment rows may map to one patient row. If the right table contains duplicate patient IDs, Pandas raises an error.
- `indicator=True`: create a `_merge` column showing whether each row matched on both sides.

For auditing, a **left join** is often preferable because it preserves every appointment and reveals unmatched patients.

In [26]:
appt_patient = appointmentsmerge(
    patients,
    how='left',
    left_on='patient_id_clean',
    right_on='patient_id_clean',
    validate='many_to_one',
    indicator=True,
    suffixes=('_appt', '_patient')
)

appt_patient['_merge'].value_counts()

_merge
both          39
left_only      1
right_only     0
Name: count, dtype: int64

## 25. Build a compact data-quality issue log

Rather than silently editing suspicious data, analysts should often create an issue log that records:
- the rule checked;
- number of affected rows;
- recommended action.

A list of dictionaries can be converted directly into a DataFrame.

In [27]:
issue_log = pd.DataFrame([
    {'rule':'Invalid appointment datetime', 'count':len(bad_appointment_dates), 'action':'Investigate / correct source'},
    {'rule':'Duplicate appointment ID', 'count':len(duplicate_appts), 'action':'Investigate duplicates'},
    {'rule':'Unknown patient ID', 'count':len(unknown_patients), 'action':'Resolve against patient master'},
    {'rule':'Unknown provider ID', 'count':len(unknown_providers), 'action':'Resolve against provider master'},
    {'rule':'Bookin after appointment', 'count':len(booking_after_appointment), 'action':'Flag as chronology error'},
    {'rule':'Checkout before check-in', 'count':len(bad_visit_times), 'action':'Flag as chronology error'},
    {'rule':'Completed visit missing timestamp', 'count':len(completed_missing_times), 'action':'Investigate incomplete workflow'},
    {'rule':'Cancelled visit with check-in', 'count':len(cancelled_with_checkin), 'action':'Investigate status/timestamp conflict'}
])

issue_log

,rule,count,action
0,Invalid appointment datetime,1,Investigate / correct source
1,Duplicate appointment ID,2,Investigate duplicates
2,Unknown patient ID,1,Resolve against patient master
3,Unknown provider ID,1,Resolve against provider master
4,Booking after appointment,1,Flag as chronology error
5,Checkout before check-in,1,Flag as chronology error
6,Completed visit missing timestamp,1,Investigate incomplete workflow
7,Cancelled visit with check-in,1,Investigate status/timestamp conflict


## 26. Save an audit output with `to_csv()`

### General syntax
```python
DataFrame.to_csv(path_or_buf, index=True, columns=None, encoding='utf-8')
```

### Important parameters
- `path_or_buf`: destination file name/path.
- `index=False`: do **not** write the DataFrame index as an extra column. Usually preferable for business datasets unless the index itself is meaningful.
- `columns=[...]`: optionally export only selected columns.
- `encoding='utf-8'`: common text encoding.

Saving an issue log is useful because it creates a reproducible audit trail.

In [28]:
issue_log.to_csv('clinic_data_quality_issue_log.csv', index=False)
print('Saved clinic_data_quality_issue_log.csv')

Saved clinic_data_quality_issue_log.csv


# In-Class Practice


### Practice 1 — Standardize visit type
The `visit_type` column contains inconsistent labels. Create `visit_type_standard` with a consistent set of values.

### Practice 2 — Check patient master uniqueness
Verify that `patient_id` is unique in the patient table. Which Pandas method would you use?

### Practice 3 — Compute scheduled lead time
Create `lead_time_days = appointment_at - booked_at`. Identify negative values.

### Practice 4 — Compute visit duration
For completed visits with valid timestamps, calculate visit duration in minutes. Flag durations less than 5 minutes or greater than 180 minutes.

### Practice 5 — Provider matching
Merge providers into the appointment table using a left join. Use `validate='many_to_one'` and `indicator=True`. How many appointments fail to match a provider?

### Practice 6 — Create your own business rule
Propose one additional validation rule that management should care about. Explain why it matters.

## Suggested syntax reminders

```python
# whole-column text normalization
df['col'].astype('string').str.strip().str.lower()

# duplicate check
df['key'].duplicated(keep=False)

# membership / referential integrity
df['key'].isin(master['key'])

# date difference in days
(df['date2'] - df['date1']).dt.total_seconds() / 86400

# time difference in minutes
(df['end'] - df['start']).dt.total_seconds() / 60

# merge with validation
left.merge(right, how='left', on='key', validate='many_to_one', indicator=True)
```

# Wrap-up: Week 2 mindset

Before analyzing data, ask:

1. **Did I load the data correctly?**
2. **Are the data types appropriate?**
3. **Are categories encoded consistently?**
4. **Are keys unique where they should be?**
5. **Do foreign keys match their master tables?**
6. **Do dates and values obey business logic?**
7. **Which problems can I safely correct, and which should I only flag?**
8. **Can I document my validation process so another analyst can reproduce it?**

The goal is not merely a DataFrame that runs without errors. The goal is a dataset that is **credible enough to support a business decision**.

In [35]:
from datetime import datetime, timedelta
import numpy as np
# Most doctors offices and clinics will see their clients if they are less than 15 minutes late so we create a condition mask

# Create condition mask 
is_late = appointments["checkin_at_dt"] > appointments["appointment_at_parsed"] + pd.Timedelta(minutes=15)

# Assign "No Show" if true, otherwise keep existing status (or set to "On Time")
appointments["status"] = np.where(is_late, "No Show", "On Time")
appointments['checkin_at_dt'] = appointments['checkin_at_dt'].fillna('No Show')

print(appointments)

   appointment_id patient_id provider_id         booked_at  \
0           A3001      P1011        D201  2026-07-17 11:15   
1           A3002      P1001        D201  2026-07-01 07:15   
2           A3003      P1008        D205  2026-07-12 06:15   
3           A3004      P1002        D204  2026-07-18 13:30   
4           A3005      P1002        D205  2026-08-05 13:15   
5           A3006      P1014        D201  2026-07-20 11:45   
6           A3007      P1010        D202  2026-07-19 08:15   
7           A3008      P1014        D201  2026-07-13 06:30   
8           A3009      P1011        D204  2026/08/01 09:15   
9           A3010      P1015        D205  2026-08-04 09:15   
10          A3011      P9999        D201  2026-07-31 07:45   
11          A3012      P1009        D201  2026-07-13 16:30   
12          A3013      P1012        D299  2026-07-28 05:00   
13          A3014      P1006        D202  2026-07-28 13:00   
14          A3015      P1004        D201  2026-07-07 09:00   
15      